<a href="https://colab.research.google.com/github/DarshiniMahesh/Chatbots-AI/blob/main/AI_Chatbot_Using_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install nltk --quiet

In [ ]:
import io
import random
import string
import warnings
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer

nltk.download('popular', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)

True

In [ ]:
# Upload chatbot.txt first using this code
from google.colab import files

uploaded = files.upload()

# Read the uploaded file
with open('chatbot.txt', 'r', encoding='utf8', errors='ignore') as f:
    raw = f.read().lower()

Saving chatbot.txt to chatbot.txt


In [ ]:
# Split corpus into sentences and words

sent_tokens = nltk.sent_tokenize(raw)   # list of sentences
word_tokens = nltk.word_tokenize(raw)   # list of words


In [ ]:
lemmer = WordNetLemmatizer()

def LemTokens(tokens):
    """Convert each token to its root/base form"""
    return [lemmer.lemmatize(token) for token in tokens]

# Dictionary that maps every punctuation character → None (removes it)
remove_punct_dict = dict((ord(punct), None) for punct in string.punctuation)

def LemNormalize(text):
    """Full pipeline: lowercase → remove punctuation → tokenize → lemmatize"""
    return LemTokens(nltk.word_tokenize(text.lower().translate(remove_punct_dict)))


In [ ]:
GREETING_INPUTS   = ("hello", "hi", "greetings", "sup", "what's up", "hey")
GREETING_RESPONSES = ["hi", "hey", "*nods*", "hi there", "hello", "I am glad! You are talking to me"]

def greeting(sentence):
    for word in sentence.split():
        if word.lower() in GREETING_INPUTS:
            return random.choice(GREETING_RESPONSES)
    return None

In [ ]:
def response(user_response):
    robo_response = ''

    # Step 1: Temporarily add user input
    sent_tokens.append(user_response)

    # Step 2: TF-IDF vectorization
    TfidfVec = TfidfVectorizer(tokenizer=LemNormalize, stop_words='english')
    tfidf    = TfidfVec.fit_transform(sent_tokens)

    # Step 3: Cosine similarity (last row = user input vs all other sentences)
    vals = cosine_similarity(tfidf[-1], tfidf)

    # Step 4: Get index of best match (2nd highest — highest is itself)
    idx  = vals.argsort()[0][-2]
    flat = vals.flatten()
    flat.sort()
    req_tfidf = flat[-2]

    # Step 5: Build response
    if req_tfidf == 0:
        robo_response = "I am sorry! I don't understand you. Try asking about chatbots, AI, or NLP."
    else:
        robo_response = sent_tokens[idx]

    # Clean up
    sent_tokens.remove(user_response)
    return robo_response

In [ ]:
flag = True

print("=" * 62)
print("🤖 ROBO: Hi! My name is Robo.")
print("🤖 ROBO: I will answer your queries about Chatbots, AI & NLP.")
print("🤖 ROBO: Type 'bye' to exit  |  'thanks' to end chat.")
print("=" * 62)
print()

while flag:
    try:
        user_response = input("👤 You: ")
    except EOFError:
        break

    user_response = user_response.lower().strip()

    if not user_response:
        continue

    if user_response == 'bye':
        flag = False
        print("🤖 ROBO: Bye! Take care.. 👋")

    elif user_response in ('thanks', 'thank you'):
        flag = False
        print("🤖 ROBO: You are welcome! 😊")

    elif greeting(user_response) is not None:
        print("🤖 ROBO:", greeting(user_response))

    else:
        print("🤖 ROBO:", response(user_response))


🤖 ROBO: Hi! My name is Robo.
🤖 ROBO: I will answer your queries about Chatbots, AI & NLP.
🤖 ROBO: Type 'bye' to exit  |  'thanks' to end chat.

👤 You: Hey
🤖 ROBO: I am glad! You are talking to me
👤 You: What is AI
🤖 ROBO: ai research has been defined as the field of study of intelligent agents, which refers to any system that perceives its environment and takes actions that maximize its chance of achieving its goals.
👤 You: Who is Alan
🤖 ROBO: in 1950, alan turing famous article computing machinery and intelligence was published, which proposed what is now called the turing test as a criterion of intelligence.
👤 You: what is NLP
🤖 ROBO: natural language processing (nlp) is a subfield of linguistics, computer science, and artificial intelligence concerned with the interactions between computers and human language.
👤 You: Bye
🤖 ROBO: Bye! Take care.. 👋
